# AcademicForge - AMD Developer Cloud Backend Setup
This notebook sets up the AcademicForge local backend (Gemma + FastAPI) on an AMD Developer Cloud instance and exposes it via ngrok so your Streamlit Community Cloud frontend can connect to it via Vercel.


### 1. Install Dependencies
Ensure you are in the AcademicForge directory and install the requirements.


In [ ]:
!pip install -r requirements.txt
# If using ROCm/AMD GPUs, ensure you install the correct PyTorch wheels here if needed.


### 2. Configure ngrok
You must provide your ngrok auth token (get it from dashboard.ngrok.com).


In [ ]:
# Replace YOUR_TOKEN_HERE with your actual ngrok token
!ngrok config add-authtoken YOUR_TOKEN_HERE


### 3. Start the Backend Server
Run the FastAPI server in the background on port 8000.


In [ ]:
import subprocess
import time
import os

# Start the uvicorn server in the background
os.environ["LOCAL_LLM_PROVIDER"] = "transformers" # or "mlx" if applicable
backend_process = subprocess.Popen(["python", "-m", "uvicorn", "backend.app:app", "--host", "0.0.0.0", "--port", "8000"])

print("Backend started in background. PID:", backend_process.pid)
time.sleep(3) # Wait a moment for it to boot


### 4. Start ngrok and Expose the API
Run ngrok in the background to expose port 8000 to the public internet.


In [ ]:
import subprocess
import urllib.request
import json
import time

# Start ngrok
ngrok_process = subprocess.Popen(["ngrok", "http", "8000", "--log=stdout"])
time.sleep(3) # Wait for ngrok to initialize

# Fetch the public URL from ngrok's local API
try:
    req = urllib.request.Request('http://127.0.0.1:4040/api/tunnels')
    res = urllib.request.urlopen(req)
    data = json.loads(res.read())
    public_url = data['tunnels'][0]['public_url']
    print("\n" + "="*60)
    print("✅ SUCCESS! Your backend is live.")
    print("👉 COPY THIS URL: ", public_url)
    print("="*60)
    print("\nNext Step: Go to your Vercel Project Settings and set BACKEND_URL to this URL. Then redeploy!")
except Exception as e:
    print("Could not retrieve ngrok URL. Ensure your token is correct and ngrok started properly.")
